# Customer Support Agent Exploration

Explore retrieval, escalation, and offline support responses independently.

In [1]:
SAMPLE_KB = [
    'Pro costs $19/mo and includes 1TB storage for 5 devices.',
    'Reset a password from the login page using Forgot Password.',
    'Cancel from Account > Subscription > Cancel. Refunds are available within 14 days.',
]
print(SAMPLE_KB)

['Pro costs $19/mo and includes 1TB storage for 5 devices.', 'Reset a password from the login page using Forgot Password.', 'Cancel from Account > Subscription > Cancel. Refunds are available within 14 days.']


## 1. Retrieve relevant knowledge

In [2]:
def retrieve(question: str, knowledge: list[str], limit: int = 2) -> list[str]:
    """Return knowledge entries containing the most question terms."""
    terms = {word.lower() for word in question.split() if len(word) > 2}
    ranked = sorted(knowledge, key=lambda item: sum(term in item.lower() for term in terms), reverse=True)
    return ranked[:limit]

In [3]:
context = retrieve('What is the Pro price?', SAMPLE_KB)
print(context)
assert context[0].startswith('Pro costs')

['Pro costs $19/mo and includes 1TB storage for 5 devices.', 'Reset a password from the login page using Forgot Password.']


## 2. Detect escalation

In [4]:
ESCALATION_TERMS = ('refund', 'fraud', 'billing error', 'data loss', 'account takeover')

def should_escalate(question: str) -> bool:
    """Route sensitive support issues to a specialist."""
    lowered = question.lower()
    return any(term in lowered for term in ESCALATION_TERMS)

In [5]:
assert should_escalate('I have a billing error')
assert not should_escalate('How do I reset my password?')
print('Escalation tests passed.')

Escalation tests passed.


## 3. Build a safe offline response

In [6]:
def demo_response(question: str, knowledge: list[str]) -> str:
    """Answer from local context without an external model call."""
    if should_escalate(question):
        return 'This request should be routed to a senior support specialist.'
    context = retrieve(question, knowledge)
    return '\n'.join(f'- {item}' for item in context) if context else 'I could not find that in the knowledge base.'

In [7]:
print(demo_response('How much is Pro?', SAMPLE_KB))
print(demo_response('I need a refund for a billing error.', SAMPLE_KB))

- Pro costs $19/mo and includes 1TB storage for 5 devices.
- Reset a password from the login page using Forgot Password.
This request should be routed to a senior support specialist.
